# Week 7 - Introduction to Data Visualization (Matplotlib)

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 7 - Introduction to Data Visualization**

---

## Session topics (from the course outline)

- Line charts, bar charts, histograms, scatter plots
- Customizing plots: titles, labels, colors

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 3** | Apply Python libraries (NumPy, Pandas) to clean, manipulate and process datasets |
| **CLO 4** | Use visualization tools to create charts, graphs and plots for effective data presentation |
| **CLO 5** | Perform exploratory data analysis (EDA) to summarize and interpret datasets |
| **CLO 7** | Analyze real-world datasets to identify trends, patterns and relationships |
| **CLO 9** | Design interactive dashboards or visual reports to support data-driven decisions |

## Reading

| Topic | Source |
|---|---|
| A brief matplotlib API primer; figures and subplots | McKinney §9.1 |
| Colors, markers, line styles; ticks, labels, legends | McKinney §9.1 |
| Saving plots to file | McKinney §9.1 |
| Choosing the right chart | Pfaffelhuber §8.1 |
| Plotting with matplotlib; line, bar, histogram, box, scatter | Pfaffelhuber §8.2–8.9 |

## Returning to Week 1

Week 1 argued, using Anscombe's quartet, that you plot the data before you trust a summary of
it. Since then you have been reading the small figures at the end of each notebook without
being told how they were made.

This session is how they were made. It covers matplotlib's object model, the four chart types
that answer most questions, and the difference between a chart that is technically correct
and one that a reader can actually use.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

---

# Part 1 - Theory and concepts

## 1.1 The object model

matplotlib has three levels of object, and almost every confusion about it comes from not
knowing which one a command is acting on.

| Object | What it is |
|---|---|
| **Figure** | The whole canvas. It holds one or more Axes, the overall title, and the size |
| **Axes** | One plot: its data area, its two axis lines, its title and labels. *Not* "the axis" |
| **Artist** | Everything drawn: lines, bars, text, the legend, the ticks |

The naming is unfortunate. An **Axes** is a single plot. An **Axis** is one of its two
directions. A figure with four panels has one Figure and four Axes.

```
   Figure  ──────────────────────────────────┐
   │  suptitle                               │
   │  ┌───────────────┐  ┌───────────────┐   │
   │  │  Axes         │  │  Axes         │   │
   │  │   title       │  │   title       │   │
   │  │   ylabel      │  │   ylabel      │   │
   │  │   [ data ]    │  │   [ data ]    │   │
   │  │   xlabel      │  │   xlabel      │   │
   │  └───────────────┘  └───────────────┘   │
   └─────────────────────────────────────────┘
```

## 1.2 Two interfaces, and why one is better

matplotlib can be driven two ways.

**The stateful (pyplot) interface** keeps a hidden "current figure" and "current axes", and
every `plt.` command acts on whichever that happens to be:

```python
plt.plot(x, y)
plt.xlabel("time")
plt.title("Readings")
```

**The object-oriented interface** names the object being acted on:

```python
fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_xlabel("time")
ax.set_title("Readings")
```

They produce identical output for a single chart. They diverge the moment there is more than
one, because "the current axes" is then ambiguous — it is whichever was created or touched
last, which is rarely what you meant.

**This course uses the object-oriented interface throughout.** Start every figure with

```python
fig, ax = plt.subplots()
```

and give every command an explicit target. The stateful shortcuts are worth recognising
because most examples online use them, but a figure built from `plt.` calls is one that cannot
easily be turned into a panel of a larger figure later.

Note the naming convention that follows from this: the setter is `ax.set_xlabel(...)`, while
the pyplot shortcut is `plt.xlabel(...)`. Nearly every `plt.something(...)` has an
`ax.set_something(...)` counterpart.

## 1.3 The four charts

Pfaffelhuber (§8.1) frames the choice by the question, not by the appearance. This is the
same table as Week 1, now with the code.

| Question | Chart | Call |
|---|---|---|
| How does a value change over an ordered variable? | Line | `ax.plot(x, y)` |
| How do categories compare on one measure? | Bar | `ax.bar(x, h)` / `ax.barh(y, w)` |
| How is one numeric variable distributed? | Histogram | `ax.hist(values, bins=...)` |
| How do two numeric variables relate? | Scatter | `ax.scatter(x, y)` |

Two rules that follow from the table and are worth stating plainly:

- **A line chart implies that the space between the points means something.** Connecting
  points that have no natural order — four faculties, say — draws a trend that does not exist.
  Categories get bars.
- **A bar chart's value axis should start at zero**, because the reader compares bar lengths.
  Truncating the axis makes a 3% difference look like a 50% one. On a line chart, where the
  reader compares slope rather than length, a truncated axis is acceptable and often better.

## 1.4 Customisation

Every element has a setter on the Axes:

| Element | Call |
|---|---|
| Axis labels | `ax.set_xlabel("...")`, `ax.set_ylabel("...")` |
| Title | `ax.set_title("...", loc="left")` |
| Limits | `ax.set_xlim(0, 100)`, `ax.set_ylim(...)` |
| Ticks | `ax.set_xticks(positions, labels)` |
| Grid | `ax.grid(axis="y", alpha=0.3)` |
| Legend | `ax.legend()` — needs `label=` on the plotting calls |
| Reference line | `ax.axhline(y)`, `ax.axvline(x)` |
| Annotation | `ax.annotate("text", xy=(x, y))` |
| Value labels on bars | `ax.bar_label(container, fmt="%.1f")` |
| Remove a frame edge | `ax.spines["top"].set_visible(False)` |

Common style arguments, accepted by most plotting calls: `color`, `alpha`, `linewidth`,
`linestyle`, `marker`, `edgecolor`, `label`, `zorder`.

## 1.5 Colour

Colour is the element most often used badly.

- **Use colour to encode something.** If every bar is a different colour but the colours mean
  nothing, they are noise. One colour for one series is usually right.
- **Avoid the rainbow (`jet`) colormap.** It is not perceptually uniform: it creates false
  boundaries where the data is smooth and hides real ones. `viridis` is the default for a
  reason.
- **Roughly one man in twelve has a red–green colour vision deficiency.** A palette that
  relies on distinguishing red from green is unreadable for them. The palette used throughout
  this course is chosen to survive that, and the ultimate check is whether the chart still
  works in greyscale.
- **Sequential, diverging, categorical.** Sequential (`Blues`, `viridis`) for a quantity that
  runs from low to high. Diverging (`RdBu`, `coolwarm`) when there is a meaningful midpoint,
  such as zero. Categorical (a fixed list) for unordered groups.

## 1.6 Several panels

```python
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 7))
axes[0, 0].plot(...)
axes[1, 1].hist(...)
fig.suptitle("...")
fig.tight_layout()
```

`plt.subplots` returns the Figure and an array of Axes. With one row or one column, the array
is one-dimensional and is indexed `axes[0]`; with both, it is two-dimensional and is indexed
`axes[row, col]`. `axes.flat` iterates over all of them regardless of shape.

`sharex=True` and `sharey=True` lock the panels to the same scale, which is what makes panels
comparable. Anscombe's quartet in Week 1 used `sharex` and `sharey` for exactly that reason:
without them the four panels would have had four different scales and the comparison would
have been meaningless.

`fig.tight_layout()` resolves overlapping labels. Call it last.

## 1.7 Saving

```python
fig.savefig("figure.png", dpi=200, bbox_inches="tight")
```

- `dpi=200` or higher for anything going into a document; the default of 100 looks soft in
  print.
- `bbox_inches="tight"` trims surrounding whitespace and, more importantly, stops a rotated
  axis label being cut off.
- `.png` for raster, `.pdf` or `.svg` for vector output that scales without blurring. A chart
  in a printed report should be vector where possible.

Call `savefig` **before** `plt.show()`. In some backends `show()` clears the figure, and you
then save a blank file.

---

---

# Part 2 - Code example

Complete and executable. §2.2 establishes the object model, §2.3–2.6 cover the four charts,
and §2.7 rebuilds one chart step by step from the default to something publishable.

## 2.1 The data

In [ ]:
def generate_campus_data(seed: int = RANDOM_SEED) -> tuple:
    """Build three related datasets for the charting examples.

    Returns
    -------
    (daily, students, faculty_summary) : tuple of pandas.DataFrame
        daily            — one row per day, for line charts
        students         — one row per student, for histograms and scatter plots
        faculty_summary  — one row per faculty, for bar charts
    """
    generator = np.random.default_rng(seed)

    # Daily library visits over one semester, with a weekly cycle and a rising
    # trend towards the examination period.
    days = pd.date_range("2026-03-02", periods=112, freq="D")
    day_index = np.arange(len(days))
    weekday = days.dayofweek.to_numpy()

    trend = 120 + day_index * 1.6
    weekly = np.where(weekday >= 5, -55, 12)          # quieter at weekends
    exam_effect = 90 * np.exp(-((day_index - 95) ** 2) / 200)
    noise = generator.normal(0, 14, len(days))

    daily = pd.DataFrame({
        "date": days,
        "visits": np.clip(trend + weekly + exam_effect + noise, 0, None).round().astype(int),
        "is_weekend": weekday >= 5,
    })

    # Students.
    n = 300
    faculties = ["Engineering", "Business", "Science", "Arts"]
    faculty = generator.choice(faculties, size=n, p=[0.35, 0.30, 0.20, 0.15])
    ability = generator.normal(0, 1, size=n)

    students = pd.DataFrame({
        "student_id": [f"S{2026000 + i}" for i in range(n)],
        "faculty": faculty,
        "year_level": generator.choice([1, 2, 3, 4], size=n, p=[0.30, 0.30, 0.25, 0.15]),
        "study_hours": np.clip(generator.normal(9, 3.2, n) + ability * 1.1, 0, None).round(1),
        "gpa": np.clip(2.6 + ability * 0.5, 0, 4).round(2),
        "attendance_pct": np.clip(78 + ability * 9 + generator.normal(0, 5, n),
                                  0, 100).round(1),
    })

    faculty_summary = students.groupby("faculty").agg(
        n=("student_id", "count"),
        mean_gpa=("gpa", "mean"),
        mean_attendance=("attendance_pct", "mean"),
    ).round(2).reset_index()

    return daily, students, faculty_summary


daily, students, faculty_summary = generate_campus_data()

print(f"daily           : {daily.shape[0]} rows  ({daily['date'].min():%d %b} "
      f"to {daily['date'].max():%d %b})")
print(f"students        : {students.shape[0]} rows")
print(f"faculty_summary : {faculty_summary.shape[0]} rows\n")
print(faculty_summary.to_string(index=False))

## 2.2 Figure and Axes

The distinction that makes everything else make sense.

In [ ]:
# plt.subplots() returns the Figure and the Axes. Always start here.
fig, ax = plt.subplots(figsize=(8, 3.5))

print("fig is a", type(fig).__name__)
print("ax  is a", type(ax).__name__)
print("\nThe Figure is the canvas. The Axes is one plot on it.")

ax.plot(daily["date"], daily["visits"], color=PALETTE[0], linewidth=1)
ax.set_title("Library visits", loc="left")
ax.set_xlabel("date")
ax.set_ylabel("visits")

plt.show()

print("\nEvery command named its target: ax.plot, ax.set_title, ax.set_xlabel.")
print("Nothing depended on which figure happened to be 'current'.")

In [ ]:
# Why the object-oriented form matters: with two panels, plt. commands become
# ambiguous. This is the bug from the Week 1 workshop.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))

axes[0].scatter(students["study_hours"], students["gpa"], s=18,
                alpha=0.5, color=PALETTE[0])
axes[0].set_title("(a) intended for the LEFT panel", loc="left")

# plt.xlabel acts on the CURRENT axes, which is the one created last -- the right.
plt.xlabel("this label was meant for the left panel")

axes[1].hist(students["gpa"], bins=20, color=PALETTE[1], edgecolor="white")
axes[1].set_title("(b) the right panel", loc="left")

plt.tight_layout()
plt.show()

print("The label landed on the right panel, because plt.xlabel acts on whichever")
print("Axes is 'current' — the last one touched.")
print("\nax.set_xlabel() on the object you mean cannot make this mistake.")

## 2.3 Line charts: change over an ordered variable

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(daily["date"], daily["visits"], color=PALETTE[0], linewidth=1,
        alpha=0.55, label="daily")

# A rolling mean makes the trend legible through the weekly cycle.
smoothed = daily["visits"].rolling(7, center=True).mean()
ax.plot(daily["date"], smoothed, color=PALETTE[3], linewidth=2.2,
        label="7-day average")

ax.set_title("Library visits rise towards the examination period", loc="left")
ax.set_xlabel("date")
ax.set_ylabel("visits per day")
ax.legend(loc="upper left", frameon=False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("The raw series alone is too noisy to read; the smoothed line alone hides")
print("how variable the days are. Drawing both, with the raw series faded,")
print("gives the reader the trend and the spread at once.")

In [ ]:
# The rule that a line implies order. Here is the same data drawn wrongly.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(faculty_summary["faculty"], faculty_summary["mean_gpa"],
             marker="o", color=PALETTE[3], linewidth=2)
axes[0].set_title("(a) WRONG: a line between categories", loc="left")
axes[0].set_ylabel("mean GPA")

axes[1].bar(faculty_summary["faculty"], faculty_summary["mean_gpa"],
            color=PALETTE[0], edgecolor="white")
axes[1].set_title("(b) Right: bars for categories", loc="left")
axes[1].set_ylabel("mean GPA")
axes[1].set_ylim(0, 4)

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print("Panel (a) draws a slope between Arts and Business, implying a progression")
print("that does not exist. The faculties have no order, so the space between")
print("them means nothing. Categories get bars.")

## 2.4 Bar charts: comparing categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Vertical, sorted, with value labels.
ordered = faculty_summary.sort_values("mean_gpa", ascending=False)
bars = axes[0].bar(ordered["faculty"], ordered["mean_gpa"],
                   color=PALETTE[0], edgecolor="white")
axes[0].bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
axes[0].set_title("(a) Mean GPA by faculty", loc="left")
axes[0].set_ylabel("mean GPA")
axes[0].set_ylim(0, 4)
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="x", alpha=0)

# Horizontal: better when the labels are long or there are many categories.
ordered_up = faculty_summary.sort_values("n")
axes[1].barh(ordered_up["faculty"], ordered_up["n"],
             color=PALETTE[2], edgecolor="white")
axes[1].set_title("(b) Students by faculty", loc="left")
axes[1].set_xlabel("students")
axes[1].grid(axis="y", alpha=0)

plt.tight_layout()
plt.show()

print("Sorting by value is almost always an improvement: the reader can rank the")
print("categories without comparing lengths by eye. Sort alphabetically only when")
print("the reader needs to look a specific category up.")

In [ ]:
# Why a bar chart's value axis must start at zero.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)

for ax, (low, high), title in zip(
    axes,
    [(0, 4), (2.4, 2.8)],
    ["(a) Honest: axis starts at 0", "(b) Misleading: axis truncated"],
):
    ax.bar(faculty_summary["faculty"], faculty_summary["mean_gpa"],
           color=PALETTE[0] if low == 0 else PALETTE[3], edgecolor="white")
    ax.set_ylim(low, high)
    ax.set_title(title, loc="left")
    ax.set_ylabel("mean GPA")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="x", alpha=0)

plt.tight_layout()
plt.show()

spread = faculty_summary["mean_gpa"].max() - faculty_summary["mean_gpa"].min()
print(f"The real spread between the highest and lowest faculty is {spread:.2f} GPA points.")
print("\nPanel (a) shows a small difference, which is the truth.")
print("Panel (b) shows the same numbers as a dramatic gap, because the reader")
print("compares bar LENGTHS and the lengths no longer start from a common zero.")
print("\nThis is the most common way a chart lies without containing a false number.")

## 2.5 Histograms: the shape of one variable

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)

for ax, bins in zip(axes, [5, 20, 80]):
    ax.hist(students["study_hours"], bins=bins, color=PALETTE[0], edgecolor="white")
    ax.set_title(f"bins = {bins}", loc="left")
    ax.set_xlabel("study hours per week")

axes[0].set_ylabel("students")
fig.suptitle("The bin count is a choice, and it changes what you see",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Five bins hides the shape. Eighty bins shows the noise in a sample of 300")
print("and hides it again. Twenty is about right here.")
print("\nThere is no correct answer, which is the point: a histogram has a")
print("parameter, and the parameter is yours. Try three values before settling.")

In [ ]:
# A histogram with the summary statistics marked on it.
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(students["study_hours"], bins=20, color=PALETTE[0],
        edgecolor="white", alpha=0.85)

mean = students["study_hours"].mean()
median = students["study_hours"].median()
ax.axvline(mean, color=PALETTE[3], linewidth=2, label=f"mean {mean:.1f}")
ax.axvline(median, color=PALETTE[2], linewidth=2, linestyle="--",
           label=f"median {median:.1f}")

ax.set_title("Weekly study hours", loc="left")
ax.set_xlabel("hours per week")
ax.set_ylabel("students")
ax.legend(frameon=False)
ax.grid(axis="x", alpha=0)

plt.tight_layout()
plt.show()

print(f"mean {mean:.2f}, median {median:.2f} — close together, so the distribution")
print("is roughly symmetric. A large gap between them would indicate skew, and")
print("would tell you which of the two to report.")

## 2.6 Scatter plots: the relationship between two variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Plain.
axes[0].scatter(students["study_hours"], students["gpa"], s=22,
                alpha=0.55, color=PALETTE[0], edgecolor="none")
axes[0].set_title("(a) Study hours against GPA", loc="left")
axes[0].set_xlabel("study hours per week")
axes[0].set_ylabel("GPA")

# With a third variable encoded as colour, and a fitted line.
for i, faculty in enumerate(sorted(students["faculty"].unique())):
    subset = students[students["faculty"] == faculty]
    axes[1].scatter(subset["study_hours"], subset["gpa"], s=22, alpha=0.7,
                    color=PALETTE[i], edgecolor="none", label=faculty)

slope, intercept = np.polyfit(students["study_hours"], students["gpa"], 1)
line_x = np.array([students["study_hours"].min(), students["study_hours"].max()])
axes[1].plot(line_x, intercept + slope * line_x, color="black",
             linewidth=1.5, linestyle="--", label="fit")

axes[1].set_title("(b) The same, coloured by faculty", loc="left")
axes[1].set_xlabel("study hours per week")
axes[1].set_ylabel("GPA")
axes[1].legend(fontsize=8, frameon=False, loc="lower right")

plt.tight_layout()
plt.show()

r = students["study_hours"].corr(students["gpa"])
print(f"Correlation: r = {r:.3f}")
print("\nPanel (b) adds a third variable as colour. It is worth doing only when")
print("the colour answers a question — here, whether the relationship differs by")
print("faculty. It does not, which is itself a finding.")

In [ ]:
# Overplotting: when there are too many points to see the density.
dense_x = rng.normal(0, 1, 8000)
dense_y = dense_x * 0.6 + rng.normal(0, 0.8, 8000)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

axes[0].scatter(dense_x, dense_y, s=20, color=PALETTE[0])
axes[0].set_title("(a) Default: a solid blob", loc="left")

axes[1].scatter(dense_x, dense_y, s=6, alpha=0.08, color=PALETTE[0],
                edgecolor="none")
axes[1].set_title("(b) Smaller, transparent points", loc="left")

hexbin = axes[2].hexbin(dense_x, dense_y, gridsize=40, cmap="Blues")
axes[2].set_title("(c) hexbin: density directly", loc="left")
fig.colorbar(hexbin, ax=axes[2], label="count")

for ax in axes:
    ax.set_xlabel("x")
axes[0].set_ylabel("y")

plt.tight_layout()
plt.show()

print("With 8,000 points the default scatter shows only the outline of the cloud;")
print("everything inside is saturated. Transparency recovers the density, and")
print("hexbin encodes it directly. Reach for one of these whenever a scatter")
print("looks like a solid shape.")

## 2.7 From default to publishable

One chart, improved in six steps. Each step names one decision.

In [ ]:
data = faculty_summary.sort_values("mean_attendance", ascending=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(data["faculty"], data["mean_attendance"])
ax.set_title("Step 0 — the default")
plt.show()

print("Everything matplotlib gives you for free, and nothing else:")
print("  - no axis labels, so the reader does not know the units")
print("  - a title that describes the chart rather than its finding")
print("  - a full box of spines around the data")
print("  - default blue, and gridlines absent")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

steps = [
    "Step 1 — label the axes",
    "Step 2 — start the value axis at zero",
    "Step 3 — a title that states the finding",
    "Step 4 — remove what carries no information",
    "Step 5 — a gridline for the value axis only",
    "Step 6 — colour that encodes something",
]

for index, ax in enumerate(axes.flat):
    step = index + 1

    # Step 6 uses colour to mark the faculty above the university average.
    if step >= 6:
        overall = students["attendance_pct"].mean()
        colours = [PALETTE[0] if v >= overall else "#C9CBD3"
                   for v in data["mean_attendance"]]
    else:
        colours = PALETTE[0]

    bars = ax.bar(data["faculty"], data["mean_attendance"],
                  color=colours, edgecolor="white")

    # Step 1: axis labels.
    if step >= 1:
        ax.set_ylabel("mean attendance (%)")

    # Step 2: a bar chart's value axis starts at zero.
    if step >= 2:
        ax.set_ylim(0, 100)

    # Step 3: a title that says what the reader should take away.
    if step >= 3:
        ax.set_title("Attendance is even across faculties", loc="left", fontsize=11)
    else:
        ax.set_title(steps[index], loc="left", fontsize=10)

    # Step 4: drop the top and right spines; they enclose nothing.
    if step >= 4:
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)

    # Step 5: a faint horizontal gridline helps the reader read values.
    if step >= 5:
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.grid(axis="x", visible=False)

    # Step 6: label the bars and mark the reference line.
    if step >= 6:
        ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
        ax.axhline(students["attendance_pct"].mean(), color=PALETTE[3],
                   linewidth=1.2, linestyle="--")
        ax.set_title(steps[index], loc="left", fontsize=10)

    ax.tick_params(axis="x", rotation=20, labelsize=8)

fig.suptitle("The same data, six decisions apart", fontsize=13,
             fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()

In [ ]:
# The finished chart, on its own.
overall = students["attendance_pct"].mean()
colours = [PALETTE[0] if v >= overall else "#C9CBD3" for v in data["mean_attendance"]]

fig, ax = plt.subplots(figsize=(8, 4.2))

bars = ax.bar(data["faculty"], data["mean_attendance"],
              color=colours, edgecolor="white", width=0.65)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)

ax.axhline(overall, color=PALETTE[3], linewidth=1.3, linestyle="--")
ax.annotate(f"university mean {overall:.1f}%",
            xy=(3.45, overall), xytext=(3.45, overall + 6),
            color=PALETTE[3], fontsize=8, ha="right")

ax.set_ylim(0, 100)
ax.set_ylabel("mean attendance (%)")
ax.set_title("Attendance varies little between faculties",
             loc="left", fontsize=12, fontweight="bold")
ax.text(0, -0.22, "Grey bars are below the university mean. n = 300 students.",
        transform=ax.transAxes, fontsize=8, color="#666666")

for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("Six decisions, none of them difficult:")
print("  label the axes | start at zero | title states the finding")
print("  remove non-data ink | grid on the value axis only | colour means something")
print("\nNone of this changed a single number. All of it changed how quickly a")
print("reader gets the point.")

## 2.8 Several panels, and saving

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# (a) line
axes[0, 0].plot(daily["date"], daily["visits"].rolling(7, center=True).mean(),
                color=PALETTE[0], linewidth=1.8)
axes[0, 0].set_title("(a) Library visits, 7-day average", loc="left")
axes[0, 0].set_ylabel("visits")

# (b) bar
ordered = faculty_summary.sort_values("mean_gpa")
axes[0, 1].barh(ordered["faculty"], ordered["mean_gpa"],
                color=PALETTE[1], edgecolor="white")
axes[0, 1].set_title("(b) Mean GPA by faculty", loc="left")
axes[0, 1].set_xlim(0, 4)
axes[0, 1].grid(axis="y", visible=False)

# (c) histogram
axes[1, 0].hist(students["attendance_pct"], bins=22, color=PALETTE[2],
                edgecolor="white")
axes[1, 0].set_title("(c) Attendance distribution", loc="left")
axes[1, 0].set_xlabel("attendance (%)")
axes[1, 0].set_ylabel("students")

# (d) scatter
axes[1, 1].scatter(students["attendance_pct"], students["gpa"], s=16,
                   alpha=0.45, color=PALETTE[4], edgecolor="none")
axes[1, 1].set_title("(d) Attendance against GPA", loc="left")
axes[1, 1].set_xlabel("attendance (%)")
axes[1, 1].set_ylabel("GPA")

fig.suptitle("Campus data: four questions, four chart types",
             fontsize=13, fontweight="bold")
fig.tight_layout()

# Save BEFORE show: some backends clear the figure on show().
from pathlib import Path
fig.savefig("week07_panels.png", dpi=150, bbox_inches="tight")
plt.show()

saved = Path("week07_panels.png")
print(f"saved {saved.name}, {saved.stat().st_size / 1024:.0f} KB")
saved.unlink()
print("(removed again, so the notebook leaves no files behind)")

---

# Part 3 - Guided workshop (in class)

Four tasks. Each has a verification cell that checks the structure of the figure you built.

Run the helper cell below first. It exists because of a genuine matplotlib subtlety: an Axes
has **three** titles - left, centre and right and `ax.get_title()` returns only the centre
one. Since this course sets titles with `loc="left"`, a check written as `ax.get_title()`
would report every one of them as missing.

In [ ]:
def get_any_title(ax) -> str:
    """Return an Axes title from whichever position it was set in.

    An Axes has three independent titles: left, center and right.
    ax.get_title() returns only the center one, so a title set with
    loc="left" reads as empty. The verification cells use this instead.
    """
    for location in ("center", "left", "right"):
        text = ax.get_title(loc=location)
        if text:
            return text
    return ""


# Demonstration of the problem this solves.
_, demo_ax = plt.subplots(figsize=(4, 1.2))
demo_ax.set_title("set with loc='left'", loc="left")
plt.close()

print("ax.get_title()            ->", repr(demo_ax.get_title()))
print("ax.get_title(loc='left')  ->", repr(demo_ax.get_title(loc="left")))
print("get_any_title(ax)         ->", repr(get_any_title(demo_ax)))

### Task 1 - A labelled line chart

**Problem.** Draw the daily visits as a line on `ax1`, with:

- an x-axis label, a y-axis label and a title,
- a horizontal reference line at the mean number of visits,
- a legend.

**Hints**

- `ax1.plot(x, y, label="...")` then `ax1.legend()`.
- `ax1.axhline(value, ...)` draws a horizontal line; give it a `label` too.

**Expected output.** A labelled line chart, followed by `Task 1 complete`.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(10, 3.6))

# TODO 1: plot visits against date, add the mean line, label everything.


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 1.
problems = []
if not ax1.lines:
    problems.append("no line was drawn — use ax1.plot(...)")
elif len(ax1.lines) < 2:
    problems.append("only one line — add the mean reference line with axhline")
if not ax1.get_xlabel():
    problems.append("no x-axis label")
if not ax1.get_ylabel():
    problems.append("no y-axis label")
if not get_any_title(ax1):
    problems.append("no title")
if ax1.get_legend() is None:
    problems.append("no legend — pass label= to the plotting calls, then call legend()")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print(f"Task 1 complete. {len(ax1.lines)} lines drawn, fully labelled.")

### Task 2 - A sorted bar chart that starts at zero

**Problem.** On `ax2`, draw mean GPA by faculty as a bar chart, with:

- the bars sorted from highest to lowest,
- the y-axis starting at 0,
- axis labels and a title,
- the value printed on each bar.

**Hints**

- Sort the frame first: `faculty_summary.sort_values("mean_gpa", ascending=False)`.
- `bars = ax2.bar(...)` then `ax2.bar_label(bars, fmt="%.2f")`.
- `ax2.set_ylim(0, 4)`.

**Expected output.** Four labelled bars in descending order, followed by `Task 2 complete`.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(7, 4))

# TODO 2: sorted bar chart, y-axis from 0, labels, title, value labels.


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 2.
problems = []
if not ax2.patches:
    problems.append("no bars were drawn — use ax2.bar(...)")
else:
    heights = [p.get_height() for p in ax2.patches]
    if len(heights) != 4:
        problems.append(f"expected 4 bars, found {len(heights)}")
    elif heights != sorted(heights, reverse=True):
        problems.append("the bars are not sorted from highest to lowest")
    bottom, _ = ax2.get_ylim()
    if bottom > 0.01:
        problems.append(f"the y-axis starts at {bottom:.2f}, not 0 — "
                        "bar lengths must be comparable from a common zero")
if not ax2.get_ylabel():
    problems.append("no y-axis label")
if not get_any_title(ax2):
    problems.append("no title")
if not ax2.texts:
    problems.append("no value labels on the bars — use ax2.bar_label(bars, ...)")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 2 complete. Sorted, zero-based, labelled.")

### Task 3 - A histogram with the centre marked

**Problem.** On `ax3`, draw a histogram of `gpa` with 20 bins, and mark both the mean and the
median with vertical lines. Label the axes, add a title and a legend.

**Hints**

- `ax3.hist(values, bins=20, ...)`.
- `ax3.axvline(value, ...)` draws a vertical line.
- Give each line a `label=` including the value, e.g. `f"mean {m:.2f}"`.

**Expected output.** A histogram with two vertical lines, followed by `Task 3 complete`.

In [ ]:
fig3, ax3 = plt.subplots(figsize=(8, 4))

# TODO 3: histogram of gpa with 20 bins, mean and median lines, labels, legend.


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 3.
problems = []
if not ax3.patches:
    problems.append("no histogram was drawn — use ax3.hist(...)")
elif len(ax3.patches) != 20:
    problems.append(f"expected 20 bins, found {len(ax3.patches)}")
if len(ax3.lines) < 2:
    problems.append(f"expected 2 vertical lines (mean and median), "
                    f"found {len(ax3.lines)}")
if not ax3.get_xlabel():
    problems.append("no x-axis label")
if not ax3.get_ylabel():
    problems.append("no y-axis label")
if not get_any_title(ax3):
    problems.append("no title")
if ax3.get_legend() is None:
    problems.append("no legend")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 3 complete.")
    print(f"  mean   {students['gpa'].mean():.3f}")
    print(f"  median {students['gpa'].median():.3f}")

### Task 4 - A two-panel figure

**Problem.** Build a figure with two panels side by side:

- **Left** (`axes4[0]`): a scatter of `attendance_pct` against `gpa`.
- **Right** (`axes4[1]`): a histogram of `study_hours`.

Label both axes on both panels, give each a title, and add an overall figure title with
`fig4.suptitle(...)`.

**Hints**

- The Axes objects are already created for you below; use `axes4[0]` and `axes4[1]`.
- Do **not** use `plt.xlabel` - it will act on the wrong panel, as demonstrated in §2.2.

**Expected output.** Two fully labelled panels, followed by `Task 4 complete`.

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(12, 4))

# TODO 4a: scatter of attendance_pct against gpa on axes4[0], fully labelled.


# TODO 4b: histogram of study_hours on axes4[1], fully labelled.


# TODO 4c: an overall title with fig4.suptitle(...)


plt.tight_layout()
plt.show()

In [ ]:
# Verification for Task 4.
problems = []

if not axes4[0].collections:
    problems.append("no scatter on the left panel — use axes4[0].scatter(...)")
if not axes4[1].patches:
    problems.append("no histogram on the right panel — use axes4[1].hist(...)")

for i, ax in enumerate(axes4):
    side = "left" if i == 0 else "right"
    if not ax.get_xlabel():
        problems.append(f"{side} panel has no x-axis label")
    if not ax.get_ylabel():
        problems.append(f"{side} panel has no y-axis label")
    if not get_any_title(ax):
        problems.append(f"{side} panel has no title")

if fig4._suptitle is None or not fig4._suptitle.get_text():
    problems.append("no figure title — use fig4.suptitle(...)")

if problems:
    print("Not complete:")
    for p in problems:
        print("  -", p)
else:
    print("Task 4 complete. Both panels labelled, figure titled.")

---

# Part 4 - Homework

Run **Restart Kernel and Run All Cells** before submitting, then push to your
`puc-data-analysis` repository.

## Problem 1 - A reusable category chart

**Problem.** Write `plot_category_summary(df, category_col, value_col, ax=None, ascending=False)`.

It draws a sorted bar chart of the **mean** of `value_col` for each value of `category_col`,
with the presentation decisions from §2.7 already applied.

**Requirements**

| Requirement | Detail |
|---|---|
| Return value | The `Axes` object that was drawn on |
| `ax=None` | Create a new figure and axes; otherwise draw on the one given |
| Bars | One per distinct category, sorted by mean value |
| `ascending` | `False` (default) sorts highest first; `True` sorts lowest first |
| Value axis | Must start at 0 |
| Labels | y-axis labelled with `value_col`, x-axis labelled with `category_col`, and a non-empty title |
| Value labels | The mean printed on each bar, to 2 decimal places |
| Spines | Top and right hidden |

**Requirements on errors**

- Raise `KeyError` if either column is absent.
- Raise `TypeError` if `value_col` is not numeric.

Do not modify the input frame, and do not call `plt.show()` inside the function.

**Hints**

- `df.groupby(category_col)[value_col].mean().sort_values(ascending=ascending)`.
- `if ax is None: _, ax = plt.subplots()`.
- `bars = ax.bar(...)` then `ax.bar_label(bars, fmt="%.2f")`.
- `pd.api.types.is_numeric_dtype` for the type check.
- Returning the `Axes` rather than the `Figure` is the convention that lets the function be
  used as one panel of a larger figure which Problem 2 does.

In [ ]:
# Problem 1 — starter code.

def plot_category_summary(df, category_col: str, value_col: str,
                          ax=None, ascending: bool = False):
    """Draw a sorted bar chart of the mean of value_col by category_col.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    category_col : str
        Column holding the categories.
    value_col : str
        Numeric column to average.
    ax : matplotlib.axes.Axes or None
        Axes to draw on. A new figure is created when None.
    ascending : bool
        Sort direction for the bars.

    Returns
    -------
    matplotlib.axes.Axes

    Raises
    ------
    KeyError
        If either column is absent.
    TypeError
        If value_col is not numeric.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete plot_category_summary()")


ax_demo = plot_category_summary(students, "faculty", "gpa")
plt.tight_layout()
plt.show()

In [ ]:
# Problem 1 — validation. All assertions must pass.
import matplotlib

sample = pd.DataFrame({
    "group": ["A", "A", "B", "B", "C", "C"],
    "score": [10.0, 20.0, 1.0, 3.0, 50.0, 60.0],
    "label": ["x", "y", "x", "y", "x", "y"],
})
# means: A = 15, B = 2, C = 55

test_ax = plot_category_summary(sample, "group", "score")

# --- return type ---
assert isinstance(test_ax, matplotlib.axes.Axes), (
    f"Must return an Axes, got {type(test_ax).__name__}. Returning the Axes is "
    "what lets the function be used as one panel of a larger figure."
)

# --- bars ---
heights = [patch.get_height() for patch in test_ax.patches]
assert len(heights) == 3, f"One bar per category, so 3; got {len(heights)}"
assert np.allclose(sorted(heights, reverse=True), [55.0, 15.0, 2.0]), (
    f"Bar heights should be the group means 55, 15, 2; got {sorted(heights, reverse=True)}"
)
assert np.allclose(heights, [55.0, 15.0, 2.0]), (
    f"With ascending=False the bars run highest to lowest; got {heights}"
)

# --- ascending=True reverses them ---
_, spare_ax = plt.subplots()
asc_ax = plot_category_summary(sample, "group", "score", ax=spare_ax, ascending=True)
asc_heights = [patch.get_height() for patch in asc_ax.patches]
assert np.allclose(asc_heights, [2.0, 15.0, 55.0]), (
    f"With ascending=True the bars run lowest to highest; got {asc_heights}"
)

# --- the ax argument is honoured ---
assert asc_ax is spare_ax, "When ax is given, draw on it and return that same Axes"

# --- presentation requirements ---
bottom, _ = test_ax.get_ylim()
assert bottom <= 0.01, f"The value axis must start at 0; it starts at {bottom}"
assert test_ax.get_ylabel(), "The y-axis must be labelled"
assert "score" in test_ax.get_ylabel(), (
    f"The y-axis label should mention the value column; got {test_ax.get_ylabel()!r}"
)
assert test_ax.get_xlabel(), "The x-axis must be labelled"
assert get_any_title(test_ax), "The chart must have a title (any of left/center/right)"
assert len(test_ax.texts) >= 3, (
    f"Each bar needs a value label; found {len(test_ax.texts)} text objects"
)
assert not test_ax.spines["top"].get_visible(), "Hide the top spine"
assert not test_ax.spines["right"].get_visible(), "Hide the right spine"

# --- the input is untouched ---
before = sample.copy()
_, throwaway = plt.subplots()
plot_category_summary(sample, "group", "score", ax=throwaway)
assert sample.equals(before), "plot_category_summary must not modify its input"

# --- errors ---
_, err_ax = plt.subplots()
for bad_cat, bad_val in [("nope", "score"), ("group", "nope")]:
    try:
        plot_category_summary(sample, bad_cat, bad_val, ax=err_ax)
    except KeyError:
        pass
    else:
        raise AssertionError(f"A missing column ({bad_cat}, {bad_val}) must raise KeyError")

try:
    plot_category_summary(sample, "group", "label", ax=err_ax)
except TypeError:
    pass
else:
    raise AssertionError("A non-numeric value_col must raise TypeError")

plt.close("all")
print("Problem 1: all assertions passed.")

---

## Problem 2 - A four-panel summary figure

**Problem.** Write `build_summary_figure(df)`, returning `(fig, axes)` for a 2×2 figure that
summarises the student data.

**The four panels, in this order:**

| Position | Panel | Content |
|---|---|---|
| `axes[0, 0]` | Bar | Mean GPA by faculty — **call `plot_category_summary` from Problem 1** |
| `axes[0, 1]` | Histogram | `attendance_pct`, 20 bins |
| `axes[1, 0]` | Scatter | `study_hours` (x) against `gpa` (y) |
| `axes[1, 1]` | Bar | Count of students per `year_level` |

**Requirements**

- Every panel has an x-axis label, a y-axis label and a title.
- The figure has a `suptitle`.
- The figure size is at least 10 by 7 inches.
- Return the tuple `(fig, axes)`, where `axes` has shape `(2, 2)`.
- Do not call `plt.show()` inside the function.

**Hints**

- `fig, axes = plt.subplots(2, 2, figsize=(12, 8))`.
- Pass `ax=axes[0, 0]` to your Problem 1 function. This is why it returns an `Axes` and takes
  one - a function that creates its own figure could not be reused here.
- For the year-level counts, `df["year_level"].value_counts().sort_index()` gives the heights
  and the labels.
- Call `fig.tight_layout()` before returning.

In [ ]:
# Problem 2 — starter code.

def build_summary_figure(df):
    """Build a 2x2 summary figure of the student data.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain faculty, gpa, attendance_pct, study_hours, year_level.

    Returns
    -------
    (matplotlib.figure.Figure, numpy.ndarray)
        The figure, and the 2x2 array of Axes.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete build_summary_figure()")


fig_demo, axes_demo = build_summary_figure(students)
plt.show()

In [ ]:
# Problem 2 — validation. All assertions must pass.
import matplotlib

fig, axes = build_summary_figure(students)

# --- structure ---
assert isinstance(fig, matplotlib.figure.Figure), "The first return value must be a Figure"
assert hasattr(axes, "shape"), "The second return value must be the array of Axes"
assert axes.shape == (2, 2), f"Expected a 2x2 array of Axes; got shape {axes.shape}"

width, height = fig.get_size_inches()
assert width >= 10 and height >= 7, (
    f"The figure should be at least 10x7 inches; got {width:.1f}x{height:.1f}. "
    "Four panels in a small figure are unreadable."
)

# --- every panel is labelled ---
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
for row, col in positions:
    ax = axes[row, col]
    assert ax.get_xlabel(), f"axes[{row}, {col}] has no x-axis label"
    assert ax.get_ylabel(), f"axes[{row}, {col}] has no y-axis label"
    assert get_any_title(ax), f"axes[{row}, {col}] has no title"

# --- the right chart type in each position ---
assert axes[0, 0].patches, "axes[0, 0] should hold a bar chart"
assert len(axes[0, 0].patches) == students["faculty"].nunique(), (
    f"axes[0, 0] should have one bar per faculty "
    f"({students['faculty'].nunique()}); got {len(axes[0, 0].patches)}"
)

assert len(axes[0, 1].patches) == 20, (
    f"axes[0, 1] should be a 20-bin histogram; found {len(axes[0, 1].patches)} patches"
)

assert axes[1, 0].collections, (
    "axes[1, 0] should hold a scatter plot — ax.scatter creates a collection"
)

assert len(axes[1, 1].patches) == students["year_level"].nunique(), (
    f"axes[1, 1] should have one bar per year level "
    f"({students['year_level'].nunique()}); got {len(axes[1, 1].patches)}"
)

# --- the bar heights in panel (1,1) are counts ---
counts = sorted(patch.get_height() for patch in axes[1, 1].patches)
expected_counts = sorted(students["year_level"].value_counts().to_numpy())
assert np.allclose(counts, expected_counts), (
    f"axes[1, 1] bar heights should be the student counts per year level; "
    f"got {counts}, expected {expected_counts}"
)

# --- panel (0,0) used the Problem 1 function, so it inherits its conventions ---
bottom, _ = axes[0, 0].get_ylim()
assert bottom <= 0.01, (
    "axes[0, 0] should start at zero — if you called plot_category_summary it "
    "does so automatically"
)
assert not axes[0, 0].spines["top"].get_visible(), (
    "axes[0, 0] should have its top spine hidden, which plot_category_summary does"
)

# --- the figure has an overall title ---
assert fig._suptitle is not None and fig._suptitle.get_text(), (
    "The figure needs a suptitle"
)

# --- it works on a subset without error ---
subset_fig, subset_axes = build_summary_figure(students.head(60))
assert subset_axes.shape == (2, 2)

plt.close("all")
print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 120–200 words.

Section 2.4 drew the same four bars twice: once with the y-axis starting at zero, and once
truncated to the range 2.4–2.8. Both charts contain exactly the same four numbers.

1. Explain why the truncated version misleads, in terms of what the reader's eye is comparing.
2. Give one situation in which truncating the value axis is **legitimate**, and say why the
   same objection does not apply.
3. A colleague sends you a bar chart with no visible axis numbers at all. What is the first
   thing you ask them for, and why?

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 120–200 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 8

Week 8 is seaborn, which sits on top of everything you learned today. It produces
distribution plots, box plots, violin plots, pair plots and heatmaps in one line each and
because every seaborn function accepts an `ax=` argument, they compose into the multi-panel
figures you built today.

Before the session, read McKinney §9.2 and Pfaffelhuber §8.10.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §9.1.

Pfaffelhuber, P. *Python for Data Analysis* (course notes). §8.1–8.9.
Available at `https://pfaffelh.github.io/python_for_data/`.